# 06 - Export Yogyakarta Deployment Models

This notebook retrains and exports the selected hazard-specific LSTM Autoencoder models for deployment:

- `rain_7d` for curah hujan tinggi
- `wind_30d` for angin kencang

Notebook 05 is for sensitivity analysis. This notebook is for creating deployable artifacts under `artifacts/deployment/`.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import random
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "yogyakarta_weather_features.csv"
DEPLOYMENT_DIR = PROJECT_ROOT / "artifacts" / "deployment"
REPORT_DIR = PROJECT_ROOT / "reports"

DEPLOYMENT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MAX_EPOCHS = 80
BATCH_SIZE = 32
PATIENCE = 10
LEARNING_RATE = 0.001

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_PATH)
print("Deployment artifacts:", DEPLOYMENT_DIR)

## 2. Load Processed Features

In [ ]:
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(
        "Run notebook 01 first: data/processed/yogyakarta_weather_features.csv was not found."
    )

data = pd.read_csv(PROCESSED_PATH, parse_dates=["date"], dtype={"station_id": "string"})
data = data.sort_values(["station_id", "date"]).reset_index(drop=True)

print("Rows:", len(data))
print("Date range:", data["date"].min(), "to", data["date"].max())
display(data.groupby("station_id")["date"].agg(["min", "max", "count"]))
display(data.head())

## 3. Deployment Model Configurations

In [ ]:
DEPLOYMENT_MODELS = [
    {
        "experiment_id": "rain_7d",
        "sequence_length": 7,
        "feature_group": "rain_only",
        "hazard": "curah_hujan_tinggi",
        "description": "Deployment model for short-term high rainfall sensitivity.",
    },
    {
        "experiment_id": "wind_30d",
        "sequence_length": 30,
        "feature_group": "wind_only",
        "hazard": "angin_kencang",
        "description": "Deployment model for strong-wind anomaly sensitivity.",
    },
]

pd.DataFrame(DEPLOYMENT_MODELS)

## 4. Helper Functions

In [ ]:
def resolve_feature_groups(data: pd.DataFrame) -> dict:
    return {
        "rain_only": ["RR", "rain_3d", "rain_7d", "rain_change_1d", "missing_RR"],
        "wind_only": [
            "ff_x",
            "ff_avg",
            "wind_change_1d",
            "ddd_x_sin",
            "ddd_x_cos",
            "missing_ff_x",
            "missing_ff_avg",
        ],
    }


def validate_required_columns(data: pd.DataFrame, feature_names: list, experiment_id: str) -> None:
    required_columns = ["date", "station_id", "station_name", "region_name"] + feature_names
    missing = [column for column in required_columns if column not in data.columns]
    if missing:
        raise ValueError(f"{experiment_id} cannot run because these columns are missing: {missing}")


def chronological_split(data: pd.DataFrame) -> tuple:
    ordered_dates = np.array(sorted(data["date"].dropna().unique()))
    if len(ordered_dates) < 30:
        raise ValueError("Not enough unique dates for chronological train/validation/test split.")

    train_end = int(len(ordered_dates) * 0.70)
    validation_end = int(len(ordered_dates) * 0.85)

    train_dates = ordered_dates[:train_end]
    validation_dates = ordered_dates[train_end:validation_end]
    test_dates = ordered_dates[validation_end:]

    train_df = data[data["date"].isin(train_dates)].copy()
    validation_df = data[data["date"].isin(validation_dates)].copy()
    test_df = data[data["date"].isin(test_dates)].copy()

    return train_df, validation_df, test_df


def fit_transform_splits(train_df, validation_df, test_df, feature_names) -> tuple:
    scaler = RobustScaler()
    scaler.fit(train_df[feature_names])

    def scale_frame(df):
        scaled = df.copy()
        scaled[feature_names] = scaler.transform(scaled[feature_names])
        return scaled

    return scale_frame(train_df), scale_frame(validation_df), scale_frame(test_df), scaler


def build_sequences(feature_df, metadata_df, feature_names, sequence_length) -> tuple:
    x_values = []
    metadata_rows = []
    skipped_windows = 0

    feature_df = feature_df.sort_values(["station_id", "date"]).reset_index(drop=True)
    metadata_df = metadata_df.sort_values(["station_id", "date"]).reset_index(drop=True)

    for station_id, meta_group in metadata_df.groupby("station_id", sort=False):
        group_index = meta_group.index
        g_meta = meta_group.reset_index(drop=True)
        g_feature = feature_df.loc[group_index].reset_index(drop=True)
        values = g_feature[feature_names].to_numpy(dtype=np.float32)

        for end_idx in range(sequence_length - 1, len(g_meta)):
            start_idx = end_idx - sequence_length + 1
            window_dates = g_meta.loc[start_idx:end_idx, "date"]
            day_differences = window_dates.diff().dt.days.iloc[1:]

            if not (day_differences == 1).all():
                skipped_windows += 1
                continue

            x_values.append(values[start_idx:end_idx + 1])
            end_row = g_meta.loc[end_idx]
            metadata_rows.append(
                {
                    "date": end_row["date"],
                    "station_id": str(end_row["station_id"]),
                    "station_name": end_row.get("station_name", ""),
                    "region_name": end_row.get("region_name", ""),
                }
            )

    if not x_values:
        empty_shape = (0, sequence_length, len(feature_names))
        return np.empty(empty_shape, dtype=np.float32), pd.DataFrame(metadata_rows), skipped_windows

    return np.stack(x_values).astype(np.float32), pd.DataFrame(metadata_rows), skipped_windows


def build_compact_lstm_autoencoder(sequence_length, n_features, learning_rate=0.001) -> Model:
    inputs = Input(shape=(sequence_length, n_features))
    encoded = LSTM(32, activation="tanh", return_sequences=True)(inputs)
    encoded = Dropout(0.15)(encoded)
    encoded = LSTM(16, activation="tanh", return_sequences=False)(encoded)

    decoded = RepeatVector(sequence_length)(encoded)
    decoded = LSTM(16, activation="tanh", return_sequences=True)(decoded)
    decoded = Dropout(0.15)(decoded)
    decoded = LSTM(32, activation="tanh", return_sequences=True)(decoded)
    outputs = TimeDistributed(Dense(n_features))(decoded)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse")
    return model


def weight_vector(feature_names: list, feature_group: str) -> np.ndarray:
    rain_weights = {"RR": 3.0, "rain_3d": 3.0, "rain_7d": 3.0, "rain_change_1d": 2.0, "missing_RR": 0.5}
    wind_weights = {
        "ff_x": 3.0,
        "ff_avg": 3.0,
        "wind_change_1d": 3.0,
        "ddd_x_sin": 1.0,
        "ddd_x_cos": 1.0,
        "missing_ff_x": 0.5,
        "missing_ff_avg": 0.5,
    }

    source = rain_weights if feature_group == "rain_only" else wind_weights
    weights = np.array([source.get(name, 1.0) for name in feature_names], dtype=np.float32)
    return weights / weights.mean()


def weighted_reconstruction_error(x_true, x_pred, weights) -> np.ndarray:
    weights = weights.reshape(1, 1, -1)
    squared_error = np.square(x_true - x_pred)
    return np.mean(squared_error * weights, axis=(1, 2))


def derive_thresholds(scores) -> dict:
    return {
        "p95": float(np.percentile(scores, 95)),
        "p99": float(np.percentile(scores, 99)),
        "p995": float(np.percentile(scores, 99.5)),
    }


feature_groups = resolve_feature_groups(data)
for config in DEPLOYMENT_MODELS:
    validate_required_columns(data, feature_groups[config["feature_group"]], config["experiment_id"])

print("Deployment feature groups are valid.")

## 5. Train and Export Deployment Models

This cell trains the two selected deployment models and writes deployment artifacts. Run it on the Jupyter server.

In [ ]:
train_df, validation_df, test_df = chronological_split(data)
export_rows = []

for config in DEPLOYMENT_MODELS:
    experiment_id = config["experiment_id"]
    sequence_length = config["sequence_length"]
    feature_group = config["feature_group"]
    feature_names = feature_groups[feature_group]
    output_dir = DEPLOYMENT_DIR / experiment_id
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print(f"Training deployment model: {experiment_id}")
    print(f"Hazard: {config['hazard']}")
    print(f"Sequence length: {sequence_length}")
    print(f"Features: {feature_names}")

    tf.keras.backend.clear_session()

    train_scaled, validation_scaled, test_scaled, scaler = fit_transform_splits(
        train_df, validation_df, test_df, feature_names
    )

    x_train, train_meta, skipped_train = build_sequences(train_scaled, train_df, feature_names, sequence_length)
    x_validation, validation_meta, skipped_validation = build_sequences(
        validation_scaled, validation_df, feature_names, sequence_length
    )
    x_test, test_meta, skipped_test = build_sequences(test_scaled, test_df, feature_names, sequence_length)

    if len(x_train) == 0 or len(x_validation) == 0:
        raise ValueError(
            f"{experiment_id} has too few consecutive rows. "
            f"Shapes: train={x_train.shape}, validation={x_validation.shape}, test={x_test.shape}"
        )

    model = build_compact_lstm_autoencoder(sequence_length, len(feature_names), LEARNING_RATE)
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        mode="min",
    )

    history = model.fit(
        x_train,
        x_train,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(x_validation, x_validation),
        callbacks=[early_stopping],
        verbose=1,
    )

    weights = weight_vector(feature_names, feature_group)
    validation_pred = model.predict(x_validation, verbose=0)
    validation_scores = weighted_reconstruction_error(x_validation, validation_pred, weights)
    thresholds = derive_thresholds(validation_scores)

    test_scores = None
    if len(x_test) > 0:
        test_pred = model.predict(x_test, verbose=0)
        test_scores = weighted_reconstruction_error(x_test, test_pred, weights)

    model_path = output_dir / "model.keras"
    scaler_path = output_dir / "scaler.pkl"
    threshold_path = output_dir / "threshold.json"
    feature_config_path = output_dir / "feature_config.json"
    training_report_path = output_dir / "training_report.json"

    model.save(model_path)
    joblib.dump(scaler, scaler_path)

    feature_config = {
        "experiment_id": experiment_id,
        "hazard": config["hazard"],
        "sequence_length": sequence_length,
        "feature_group": feature_group,
        "model_features": feature_names,
        "feature_weights": {name: float(weight) for name, weight in zip(feature_names, weights)},
        "source_notebook": "06_yogyakarta_export_deployment_models.ipynb",
        "notes": "Deployment artifact trained from processed DI Yogyakarta daily features.",
    }

    training_report = {
        "experiment_id": experiment_id,
        "hazard": config["hazard"],
        "train_sequences": int(len(x_train)),
        "validation_sequences": int(len(x_validation)),
        "test_sequences": int(len(x_test)),
        "skipped_train_windows": int(skipped_train),
        "skipped_validation_windows": int(skipped_validation),
        "skipped_test_windows": int(skipped_test),
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "stopped_epoch": int(len(history.history["loss"])),
        "thresholds": thresholds,
        "validation_score_min": float(np.min(validation_scores)),
        "validation_score_max": float(np.max(validation_scores)),
        "test_score_min": float(np.min(test_scores)) if test_scores is not None else None,
        "test_score_max": float(np.max(test_scores)) if test_scores is not None else None,
        "loss_history": {key: [float(value) for value in values] for key, values in history.history.items()},
    }

    threshold_path.write_text(json.dumps(thresholds, indent=2), encoding="utf-8")
    feature_config_path.write_text(json.dumps(feature_config, indent=2), encoding="utf-8")
    training_report_path.write_text(json.dumps(training_report, indent=2), encoding="utf-8")

    export_rows.append(
        {
            "experiment_id": experiment_id,
            "hazard": config["hazard"],
            "sequence_length": sequence_length,
            "feature_group": feature_group,
            "n_features": len(feature_names),
            "train_sequences": int(len(x_train)),
            "validation_sequences": int(len(x_validation)),
            "test_sequences": int(len(x_test)),
            "best_val_loss": float(np.min(history.history["val_loss"])),
            "stopped_epoch": int(len(history.history["loss"])),
            "threshold_p95": thresholds["p95"],
            "threshold_p99": thresholds["p99"],
            "threshold_p995": thresholds["p995"],
            "artifact_dir": str(output_dir.relative_to(PROJECT_ROOT)),
        }
    )

    print("Saved deployment artifacts to:", output_dir)

export_summary = pd.DataFrame(export_rows)
summary_path = REPORT_DIR / "deployment_model_export_summary.csv"
export_summary.to_csv(summary_path, index=False)

display(export_summary)
print("Saved summary:", summary_path)

## 6. Verification

In [ ]:
expected_files = []
for config in DEPLOYMENT_MODELS:
    model_dir = DEPLOYMENT_DIR / config["experiment_id"]
    expected_files.extend(
        [
            model_dir / "model.keras",
            model_dir / "scaler.pkl",
            model_dir / "threshold.json",
            model_dir / "feature_config.json",
            model_dir / "training_report.json",
        ]
    )

expected_files.append(REPORT_DIR / "deployment_model_export_summary.csv")

verification = pd.DataFrame(
    {
        "path": [str(path.relative_to(PROJECT_ROOT)) for path in expected_files],
        "exists": [path.exists() for path in expected_files],
    }
)

display(verification)

missing_outputs = verification[~verification["exists"]]
if not missing_outputs.empty:
    raise AssertionError("Some deployment artifact files were not created.")

print("Deployment model export verification passed.")

## Deployment Note

These artifacts are trained from the historical daily Yogyakarta feature table. Before connecting to IoT production data, confirm that IoT sensor fields can produce the same features. If the deployed IoT device does not measure wind direction, retrain the wind model without `ddd_x_sin` and `ddd_x_cos` before production use.